# 3.4: Linear Regression Implementation from Scratch

In this section, we will implement the entire emthod from scratch, including: (i) the model; (ii) the loss function; (iii) a minibatch stochastic gradient descent optimizer, and (iv) the training function that stitches all of these pieces together. Finally, we will run our synthetic data generator from Section 3.3 and apply our model on the resulting dataset.

We will rely only on tensors and automatic differentiation for this section. Later, we will introduce a more concise implementation, taking advantage of the bells and whistles of deep learning frameworks while retaining the structure of what follows below.

In [ ]:
%matplotlib inline
import torch
from d2l import torch as d2l

## 3.4.1: Defining the model

Start by initializing weights by drawing random numbers from a normal distrbution with $\mu=0$ and $\sigma=0.01$. These numbers often work well in practice. We also set the bias to 0.

In [ ]:
class LinearRegressionScratch(d2l.Module): #@save
    """The linear regression model implemented from scratch."""
    def __init__(self, num_inputs, lr, sigma=0.01):
        super().__init__()
        self.save_hyperparameters()
        self.w = torch.normal(0, sigma, (num_inputs, 1), requires_grad=True) # mean, std, shape (vector in \mathbb{R}^{N}), require_gradient
        self.b = torch.zeros(1, requires_grad=True) # remember: y = \vec{w}^T\vec{x}+b
                                                    # this is the value of y_i for \vec{y} = \bold{X}\vec{w}+b, where b is broadcasted from a scalar into a vector

## 3.4.2: Defining the Loss Function

In [ ]:
@d2l.add_to_class(LinearRegressionScratch)
def loss(self, y_hat, y):
    l = (y_hat - y) ** 2 / 2 # Calculate the vector of MSE while preserving the assumption that differences are normally distributed
    return l.mean()

## 3.4.3: Defining the Optimization Algorithm

Although linear regression has a closed-form solution, we will use minbatch Stochastic Gradient Descent (SGD) for this problem.

At each step, we randomly sample a minibatch from the dataset, estimate gradient of loss WRT parameters, and update parameters in the direction that may reduce the loss.